In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import mlflow
import mlflow.tensorflow

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [4]:
train_data = train_datagen.flow_from_directory(
    "dataset",
    target_size=(128,128),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_data = train_datagen.flow_from_directory(
    "dataset",
    target_size=(128,128),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

Found 81 images belonging to 2 classes.
Found 20 images belonging to 2 classes.


In [5]:
model = models.Sequential([

    layers.Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(128,128,3)
    ),

    layers.MaxPooling2D((2,2)),

    layers.Conv2D(
        64,
        (3,3),
        activation='relu'
    ),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(
        64,
        activation='relu'
    ),
    layers.Dense(
        1,
        activation='sigmoid'
    )
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\cbina\OneDrive\Desktop\Img Classification\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 57600)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │       3,686,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,705,921 (14.14 MB)

 Trainable params: 3,705,921 (14.14 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(
    "Binary_Image_Classification"
)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779440642196, experiment_id='2', last_update_time=1779440642196, lifecycle_stage='active', name='Binary_Image_Classification', tags={}, trace_location=None, workspace='default'>

In [9]:
EPOCHS = 4
BATCH_SIZE = 16

mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(
    "Binary_Image_Classification"
)

with mlflow.start_run():

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS
    )

    # Log parameters
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)

    # Log metrics
    mlflow.log_metric(
        "accuracy",
        history.history['accuracy'][-1]
    )

    mlflow.log_metric(
        "val_accuracy",
        history.history['val_accuracy'][-1]
    )

    mlflow.log_metric(
        "loss",
        history.history['loss'][-1]
    )

    mlflow.log_metric(
        "val_loss",
        history.history['val_loss'][-1]
    )

    # Log model
    mlflow.tensorflow.log_model(
        model,
        name="model"
    )

print("Experiment Completed")

Epoch 1/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 69s 13s/step - accuracy: 0.9877 - loss: 0.1619 - val_accuracy: 0.5000 - val_loss: 0.9571
Epoch 2/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.9753 - loss: 0.1067 - val_accuracy: 0.5000 - val_loss: 0.9818
Epoch 3/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9877 - loss: 0.0866 - val_accuracy: 0.4500 - val_loss: 1.4523
Epoch 4/4
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 743ms/step - accuracy: 1.0000 - loss: 0.0469 - val_accuracy: 0.5500 - val_loss: 1.1822


2026/05/22 17:05:08 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run indecisive-vole-101 at: http://127.0.0.1:5000/#/experiments/2/runs/8e57fce2acbd45c6b24307a85d0f63fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Experiment Completed
